In [1]:
import json, random, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RUN_FULL_LLM = False
print("RUN_FULL_LLM:", RUN_FULL_LLM)

RUN_FULL_LLM: False


# 53 - Classification Fine-Tuning Lab: Prompting vs Fine-Tuning

## Purpose
Answer the practical ML question: when should you fine-tune for classification, and when is prompting enough?

## Experimental Setup
| Approach | Method | Labeled Data Needed |
|----------|--------|-----------------------|
| Zero-Shot NLI | HF cross-encoder NLI pipeline | None |
| Fine-Tuned Classifier | TF-IDF + Logistic Regression | All train examples |
| LLM SFT (optional) | LoRA on Qwen2.5-0.5B | All train examples |

In [2]:
INTENTS = ["billing", "technical_issue", "account_access",
           "feature_request", "cancellation", "general_inquiry"]

raw_examples = [
    ("I was charged twice this month", "billing"),
    ("Why is my invoice higher than usual?", "billing"),
    ("Can I get a receipt for last month payment?", "billing"),
    ("The payment failed but my card was charged", "billing"),
    ("I need to update my credit card information", "billing"),
    ("When will I be charged for the next cycle?", "billing"),
    ("Is there a discount for annual billing?", "billing"),
    ("My refund has not appeared yet", "billing"),
    ("The app crashes when I upload files", "technical_issue"),
    ("Dashboard is loading very slowly today", "technical_issue"),
    ("Getting a 500 error on the API", "technical_issue"),
    ("The export feature is not working", "technical_issue"),
    ("Data sync is stuck at 50 percent", "technical_issue"),
    ("Charts are showing incorrect numbers", "technical_issue"),
    ("The mobile view is completely broken", "technical_issue"),
    ("Webhook notifications stopped working", "technical_issue"),
    ("I cannot log in to my account", "account_access"),
    ("Forgot password and reset email is not coming", "account_access"),
    ("My account was locked after too many attempts", "account_access"),
    ("How do I enable two-factor authentication?", "account_access"),
    ("I need to change the email on my account", "account_access"),
    ("Can I merge my two accounts into one?", "account_access"),
    ("It would be great to have dark mode", "feature_request"),
    ("Can you add a Slack integration?", "feature_request"),
    ("I wish I could customize the report templates", "feature_request"),
    ("Do you plan to support SSO login?", "feature_request"),
    ("Could you add a Gantt chart view?", "feature_request"),
    ("It would help to have keyboard shortcuts", "feature_request"),
    ("I want to cancel my subscription", "cancellation"),
    ("How do I close my account permanently?", "cancellation"),
    ("I am switching to a competitor", "cancellation"),
    ("Please stop charging me I no longer use this", "cancellation"),
    ("What happens to my data if I cancel?", "cancellation"),
    ("What is the difference between Pro and Enterprise?", "general_inquiry"),
    ("Do you have any tutorials or documentation?", "general_inquiry"),
    ("How many users can I add on my plan?", "general_inquiry"),
    ("Is there a free trial available?", "general_inquiry"),
    ("What are your support hours?", "general_inquiry"),
    ("Do you offer on-premise deployment?", "general_inquiry"),
]

from collections import defaultdict
by_intent = defaultdict(list)
for text, label in raw_examples:
    by_intent[label].append((text, label))

train_set, test_set = [], []
for intent, examples in by_intent.items():
    train_set.extend(examples[:-2])
    test_set.extend(examples[-2:])

train_texts = [x[0] for x in train_set]
train_labels = [x[1] for x in train_set]
test_texts = [x[0] for x in test_set]
test_labels = [x[1] for x in test_set]

print(f"Train: {len(train_set)} | Test: {len(test_set)} | Intents: {len(INTENTS)}")

Train: 27 | Test: 12 | Intents: 6


## Approach 1: Zero-Shot (Keyword Similarity — No Labeled Data)

Uses TF-IDF cosine similarity between the test text and hand-crafted **label keyword descriptions**.
No training examples are used — this mirrors the "prompting" paradigm:
you describe what each class means in natural language and let the system classify.

**Concept**: Fit a TF-IDF space on label descriptions only, then classify by nearest prototype.
- Pro: no labeled data needed  
- Con: brittle, vocabulary-dependent, no context learning


In [3]:
# Zero-shot: TF-IDF cosine similarity to label descriptions (no labeled data)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

LABEL_DESCRIPTIONS = [
    "billing payment invoice charge refund",
    "technical error bug crash slow not working",
    "account login password access locked reset",
    "feature request suggestion improvement add",
    "cancel cancellation close account subscription",
    "general question inquiry information help",
]

# Fit vectorizer on label descriptions only - no training labels used
_zs_vec = TfidfVectorizer(ngram_range=(1,2), stop_words="english")
_zs_vec.fit(LABEL_DESCRIPTIONS)
_proto = _zs_vec.transform(LABEL_DESCRIPTIONS)

def classify_zeroshot(text):
    vec = _zs_vec.transform([text])
    sims = cosine_similarity(vec, _proto)[0]
    return INTENTS[sims.argmax()]

t0 = time.time()
zs_preds = [classify_zeroshot(t) for t in test_texts]
zs_time = time.time() - t0
zs_correct = sum(p == g for p, g in zip(zs_preds, test_labels))
zs_acc = zs_correct / len(test_labels) * 100

print(f"Zero-Shot Accuracy: {zs_acc:.1f}% ({zs_correct}/{len(test_labels)}) in {zs_time:.3f}s")


Zero-Shot Accuracy: 41.7% (5/12) in 0.007s


## Approach 2: Fine-Tuned Classifier (TF-IDF + Logistic Regression)

Train a supervised classifier on all labeled examples.
Represents the lightest-weight credible fine-tuning approach.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

ft_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED, C=5.0)),
])

t0 = time.time()
ft_pipeline.fit(train_texts, train_labels)
ft_train_time = time.time() - t0

t1 = time.time()
ft_preds = ft_pipeline.predict(test_texts)
ft_infer_time = time.time() - t1

ft_acc = accuracy_score(test_labels, ft_preds) * 100
print(f"Fine-Tuned Accuracy: {ft_acc:.1f}% ({sum(p==g for p,g in zip(ft_preds,test_labels))}/{len(test_labels)})")
print(f"Train time: {ft_train_time:.3f}s  Inference: {ft_infer_time*1000:.1f}ms\n")
print(classification_report(test_labels, ft_preds, target_names=INTENTS))

Fine-Tuned Accuracy: 50.0% (6/12)
Train time: 0.017s  Inference: 0.7ms

                 precision    recall  f1-score   support

        billing       0.00      0.00      0.00         2
technical_issue       0.00      0.00      0.00         2
 account_access       1.00      0.50      0.67         2
feature_request       1.00      1.00      1.00         2
   cancellation       0.50      0.50      0.50         2
general_inquiry       0.50      1.00      0.67         2

       accuracy                           0.50        12
      macro avg       0.50      0.50      0.47        12
   weighted avg       0.50      0.50      0.47        12



## Optional: LLM SFT (LoRA)

Set `RUN_FULL_LLM = True` to run LoRA fine-tuning on Qwen2.5-0.5B-Instruct.

In [5]:
llm_acc = None

if RUN_FULL_LLM:
    try:
        import torch
        from datasets import Dataset
        from peft import LoraConfig, AutoPeftModelForCausalLM
        from trl import SFTTrainer, SFTConfig
        from transformers import AutoTokenizer

        MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
        OUT = str(ARTIFACT_DIR / "intent_lora")
        SYS = f"Classify into one of: {', '.join(INTENTS)}. Output only the label."

        train_ds = Dataset.from_list([
            {"messages": [{"role": "system", "content": SYS}, {"role": "user", "content": t}, {"role": "assistant", "content": l}]}
            for t, l in train_set
        ])
        trainer = SFTTrainer(
            model=MODEL_ID,
            args=SFTConfig(output_dir=OUT, max_steps=60, per_device_train_batch_size=4,
                           gradient_accumulation_steps=2, learning_rate=2e-4, warmup_steps=5,
                           report_to="none", seed=SEED, bf16=torch.cuda.is_available()),
            train_dataset=train_ds,
            peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                   target_modules=["q_proj","k_proj","v_proj","o_proj"],
                                   task_type="CAUSAL_LM"),
        )
        trainer.train()
        trainer.save_model(OUT)

        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        llm_model = AutoPeftModelForCausalLM.from_pretrained(OUT, device_map="auto",
                                                              torch_dtype=torch.bfloat16)

        def classify_llm(text):
            msgs = [{"role":"system","content":SYS},{"role":"user","content":text}]
            p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = tok(p, return_tensors="pt").to(llm_model.device)
            with torch.no_grad():
                out = llm_model.generate(**inp, max_new_tokens=15, do_sample=False)
            raw = tok.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True).strip().lower()
            for intent in INTENTS:
                if intent in raw: return intent
            return raw

        llm_preds = [classify_llm(t) for t in test_texts]
        llm_acc = accuracy_score(test_labels, llm_preds) * 100
        print(f"LLM SFT Accuracy: {llm_acc:.1f}%")
    except Exception as e:
        print(f"LLM SFT skipped: {type(e).__name__}: {e}")
else:
    print("RUN_FULL_LLM=False — LLM SFT skipped.")

RUN_FULL_LLM=False — LLM SFT skipped.


## Head-to-Head Comparison

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

comparison = {
    "Zero-Shot NLI": {"accuracy": zs_acc, "train_examples": 0, "latency_ms": zs_time/len(test_labels)*1000},
    "TF-IDF + LR": {"accuracy": ft_acc, "train_examples": len(train_set), "latency_ms": ft_infer_time/len(test_labels)*1000},
}
if llm_acc is not None:
    comparison["LLM SFT (LoRA)"] = {"accuracy": llm_acc, "train_examples": len(train_set), "latency_ms": None}

comp_df = pd.DataFrame(comparison).T.reset_index().rename(columns={"index": "approach"})

print(f"{'Approach':<22} {'Accuracy':>10} {'Train Exs':>10} {'Lat (ms)':>10}")
print("-" * 58)
for _, row in comp_df.iterrows():
    lat = f"{row['latency_ms']:.1f}" if row["latency_ms"] is not None else "N/A"
    print(f"{row['approach']:<22} {row['accuracy']:>9.1f}% {int(row['train_examples']):>10} {lat:>10}")

comp_df.to_csv(ARTIFACT_DIR / "comparison_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 3))
colors = ["#6baed6", "#74c476", "#fd8d3c"][:len(comp_df)]
bars = ax.barh(comp_df["approach"], comp_df["accuracy"], color=colors)
ax.set_xlabel("Accuracy (%)")
ax.set_xlim(0, 115)
ax.set_title("Prompting vs Fine-Tuning — Intent Classification")
for bar, val in zip(bars, comp_df["accuracy"]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=10)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "comparison_chart.png", dpi=120)
plt.show()
print("Saved: comparison_chart.png")

Approach                 Accuracy  Train Exs   Lat (ms)
----------------------------------------------------------
Zero-Shot NLI               41.7%          0        0.6
TF-IDF + LR                 50.0%         27        0.1
Saved: comparison_chart.png


In [7]:
pcdf = pd.DataFrame(index=INTENTS)

for intent in INTENTS:
    gold_bin = [l == intent for l in test_labels]
    pcdf.loc[intent, "count"] = sum(gold_bin)
    pcdf.loc[intent, "zero_shot_%"] = round(sum(p==g and g for p,g in zip([pz==intent for pz in zs_preds], gold_bin)) / max(sum(gold_bin),1) * 100, 1)
    pcdf.loc[intent, "fine_tuned_%"] = round(sum(p==g and g for p,g in zip([pf==intent for pf in ft_preds], gold_bin)) / max(sum(gold_bin),1) * 100, 1)

pcdf["winner"] = pcdf.apply(
    lambda r: "fine-tuned" if r["fine_tuned_%"] > r["zero_shot_%"]
    else ("tie" if r["fine_tuned_%"] == r["zero_shot_%"] else "zero-shot"), axis=1
)
pcdf.to_csv(ARTIFACT_DIR / "per_class_analysis.csv")
display(pcdf)

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(INTENTS))
w = 0.35
ax.bar([i - w/2 for i in x], pcdf["zero_shot_%"], width=w, label="Zero-Shot", color="#6baed6")
ax.bar([i + w/2 for i in x], pcdf["fine_tuned_%"], width=w, label="Fine-Tuned", color="#74c476")
ax.set_xticks(list(x))
ax.set_xticklabels([i.replace("_"," ") for i in INTENTS], rotation=25, ha="right")
ax.set_ylabel("Accuracy per class (%)")
ax.set_ylim(0, 115)
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Per-Class Accuracy: Zero-Shot vs Fine-Tuned")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "per_class_chart.png", dpi=120)
plt.show()
print("Saved: per_class_chart.png")

,count,zero_shot_%,fine_tuned_%,winner
billing,2.0,100.0,0.0,zero-shot
technical_issue,2.0,50.0,100.0,fine-tuned
account_access,2.0,0.0,0.0,tie
feature_request,2.0,50.0,100.0,fine-tuned
cancellation,2.0,50.0,50.0,tie
general_inquiry,2.0,0.0,50.0,fine-tuned


Saved: per_class_chart.png


## Interpretation

### What the results mean
- **Zero-shot wins or ties** on most classes: prompting is sufficient; no need for labeled data
- **Fine-tuned wins on specific classes**: those classes likely need term exposure from examples
- **Large accuracy gap**: collect more data and fine-tune

### Decision guide
| Situation | Recommendation |
|-----------|---------------|
| < 50 labeled examples | Zero-shot prompting |
| 50-200 examples | Few-shot or lightweight fine-tuning |
| 200+ examples, stable schema | Fine-tune — better accuracy and lower latency |
| Domain-specific vocabulary | Fine-tune — models must learn the terms |
| Frequently changing categories | Prompting — avoid retraining overhead |

### Common mistakes
- Reporting only overall accuracy — per-class analysis shows where each approach actually fails
- Ignoring latency — zero-shot NLI can be 100x slower per sample than a fine-tuned LR model
- Fine-tuning without a holdout set — always keep examples the model never saw during training